# Latent Decorrelation Block + VolReg — Protocole propre et reproductible

**But.** Ce notebook corrige trois problèmes de la version précédente et rend l'étude défendable
en revue :

1. **La méthode doit correspondre à son nom.** Le bloc implémente ici une *vraie* décorrélation
   de canaux (blanchiment), avec la covariance estimée sur les `N = D·H·W` positions spatiales
   (et non plus un produit externe de rang 1). Un commutateur `mode` permet d'isoler proprement
   chaque composant.
2. **Le « condition number » est réel.** On journalise la *vraie* valeur propre max/min
   ($\kappa = \lambda_{\max}/\lambda_{\min}$) de la matrice effectivement inversée, pas un proxy
   `std.max()/std.min()`. L'hypothèse « blanchiment mal conditionné ⇒ frontières dégradées »
   devient ainsi testable.
3. **Statistiques honnêtes.** Tests appariés (Wilcoxon), tailles d'effet appariées (Cohen *d*),
   IC bootstrap sur la médiane, et **correction multiple (Holm + Benjamini–Hochberg)** sur toute
   la famille de tests.

**Distinction établi / hypothèse.**
- *Établi* : le résultat négatif de VolReg (sur une implémentation numériquement saine).
- *À tester* : le bloc de décorrélation aide-t-il *vraiment* (vs une simple standardisation +
  attention), pour quel type de corruption, et l'effet survit-il à la correction multiple ?

> Mettre `SMOKE = True` pour un test bout-en-bout sur données synthétiques (CPU, ~1 min).
> Mettre `SMOKE = False` et renseigner les chemins du cache pour les vraies expériences.


---
**v2 (révisée après données réelles).** Cette version : sépare les ASSD *plafonnés* (=16, masques vides = échecs) des ASSD *finis* et reporte le **taux d'échec** comme métrique de premier rang ; fournit un **moteur de comparaison unifié** (`whiten` vs `Baseline`, `whiten` vs `standardize`, `standardize` vs `Baseline`) avec P5, Cohen $d$, Holm/BH ; ajoute une **table de dissociation VolReg** (Antérieur vs Postérieur) ; rend le **diagnostic de conditionnement honnête** (Spearman sur ASSD finis + échec-vs-$\gamma$, avec mention du confond $\kappa\leftrightarrow 1/\gamma$) ; et **exporte des tableaux LaTeX** (`elsarticle`/booktabs).

In [ ]:
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from scipy import ndimage
from scipy.stats import wilcoxon
from dataclasses import dataclass, field
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
try:
    from medpy.metric.binary import assd as _medpy_assd
    HAVE_MEDPY = True
except Exception:
    HAVE_MEDPY = False
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu', '| medpy:', HAVE_MEDPY)

## 1. Configuration

In [ ]:
SMOKE = True   # True: synthetic smoke test (CPU). False: real Decathlon cache.

@dataclass
class Config:
    # --- real-data paths (used only when SMOKE=False) ---
    cache_root: Path = Path('/content/drive/MyDrive/cache_Decathlon_Anterior_Posterior_32_32_32')
    out_root:   Path = Path('/content/drive/MyDrive/LOP_clean')
    target_shape: tuple = (32, 32, 32)
    # raw-label -> class index (0 bg, 1 anterior, 2 posterior). ADAPT to your cache.
    label_map: dict = field(default_factory=lambda: {0: 0, 1: 1, 2: 2})

    epochs: int = 80
    batch_size: int = 4
    lr: float = 3e-4
    weight_decay: float = 1e-4
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    seed: int = SEED

    # controlled hyperparameter of the whitening: shrinkage intensity gamma
    # (small gamma -> near rank-deficient -> large TRUE kappa). This *is* the
    # conditioning knob, replacing the meaningless eps proxy.
    gamma_list: list = field(default_factory=lambda: [1e-1, 1e-2, 1e-3, 1e-4, 1e-5])
    noise_levels: list = field(default_factory=lambda: [0.00, 0.05, 0.10, 0.15])
    blur_sigmas:  list = field(default_factory=lambda: [0.0, 0.5, 1.0, 1.5])
    alpha_volreg: float = 0.1

cfg = Config()
if SMOKE:
    cfg.epochs, cfg.batch_size, cfg.target_shape = 2, 2, (16, 16, 16)
    cfg.gamma_list = [1e-1, 1e-5]   # 2 levels in smoke just to populate the conditioning diagnostic
    cfg.noise_levels, cfg.blur_sigmas = [0.0, 0.10], [0.5]
print(cfg)

## 2. Data

Synthetic generator for smoke tests, plus a template loader matching the real cache
(`.npz` with `img` uint8 and `lab` raw labels). Replace `load_real_sets()` internals
with your split logic; the rest of the notebook is data-agnostic.

In [ ]:
def make_volume(rng, shape):
    Z, Y, X = np.indices(shape).astype(np.float32)
    lab = np.zeros(shape, np.int64)
    for cls, fx in [(1, 0.35), (2, 0.65)]:
        cx, cy, cz = shape[2]*fx, shape[1]*0.5+rng.normal(0,1), shape[0]*0.5+rng.normal(0,1)
        r = shape[0]*0.18 + rng.normal(0, 0.5)
        lab[((X-cx)**2+(Y-cy)**2+(Z-cz)**2) < r**2] = cls
    img = (lab>0).astype(np.float32)*0.7 + rng.normal(0, 0.05, shape).astype(np.float32)
    return np.clip(img, 0, 1), lab

def synth_set(n, seed):
    rng = np.random.default_rng(seed)
    return [make_volume(rng, cfg.target_shape) for _ in range(n)]

def load_real_sets():
    # TEMPLATE — adapt to your cache. Return (train_list, test_list) of (img,lab) arrays.
    files = sorted(cfg.cache_root.glob('*.npz'))
    n_test = max(1, int(0.10*len(files)))
    def load(fs):
        out = []
        for f in fs:
            d = np.load(f)
            img = d['img'].astype(np.float32)/255.0
            raw = d['lab'].astype(np.int64)
            lab = np.zeros_like(raw)
            for v, k in cfg.label_map.items(): lab[raw == v] = k
            out.append((img, lab))
        return out
    return load(files[n_test:]), load(files[:n_test])

if SMOKE:
    train_set, test_set = synth_set(8, 1), synth_set(10, 2)
else:
    train_set, test_set = load_real_sets()
print('train', len(train_set), 'test', len(test_set))

In [ ]:
def corrupt_volume(img, kind, sev, rng):
    if kind == 'noise':
        return np.clip(img + sev*rng.normal(0, 1, img.shape).astype(np.float32), 0, 1)
    if kind == 'blur':
        return ndimage.gaussian_filter(img, sigma=sev)
    return img

## 3. Latent Decorrelation (Whitening) Block — the real thing

At the bottleneck $b\in\mathbb{R}^{B\times C\times D\times H\times W}$ the $N=DHW$ spatial
positions are observations of $C$-dim channel vectors:

$$\Sigma = \tfrac{1}{N-1}\,X_c X_c^\top,\qquad
\Sigma_s = (1-\gamma)\,\Sigma + \gamma\,\bar\tau\,I,\qquad
X_w = L^{-1}X_c,\ \ \Sigma_s = LL^\top .$$

`mode` isolates each component so the ablation is unambiguous:
- `whiten` — full channel decorrelation (the method);
- `standardize` — diagonal only (≡ what the previous "LOP" actually did);
- `identity` — gating only (no normalization).

When $C>N$ (true at the bottleneck), $\Sigma$ is rank-deficient; the trace-relative
shrinkage $\gamma$ makes $\Sigma_s$ SPD and sets a *measurable* conditioning, with the
**true** $\kappa(\Sigma_s)=\lambda_{\max}/\lambda_{\min}$ returned for the diagnostic.

In [ ]:
class LatentDecorrelationBlock(nn.Module):
    def __init__(self, C, gamma=1e-2, mode='whiten'):
        super().__init__()
        self.C, self.gamma, self.mode = C, gamma, mode
        h = max(C//4, 1)
        self.attention = nn.Sequential(nn.Conv3d(C, h, 1), nn.ReLU(),
                                       nn.Conv3d(h, C, 1), nn.Sigmoid())

    def forward(self, x, return_cond=False):
        B, C, D, H, W = x.shape; N = D*H*W
        X = x.reshape(B, C, N); Xc = X - X.mean(-1, keepdim=True); kappa = None
        if self.mode == 'identity':
            feat = X
        elif self.mode == 'standardize':
            feat = Xc / torch.sqrt(Xc.var(-1, keepdim=True, unbiased=False) + self.gamma)
        else:  # whiten
            S = torch.bmm(Xc, Xc.transpose(1, 2)) / (N - 1)
            tau = torch.diagonal(S, dim1=-2, dim2=-1).mean(-1).view(B, 1, 1)
            I = torch.eye(C, device=x.device).unsqueeze(0)
            Ss = (1 - self.gamma)*S + self.gamma*tau*I
            L = torch.linalg.cholesky(Ss)
            feat = torch.linalg.solve_triangular(L, Xc, upper=False)
            if return_cond:
                with torch.no_grad():
                    ev = torch.linalg.eigvalsh(Ss)
                    kappa = (ev[:, -1] / ev[:, 0].clamp_min(1e-12)).mean().item()
        gate = self.attention(feat.reshape(B, C, D, H, W))
        out = x * gate                      # scale-preserving gating
        return (out, kappa) if return_cond else out

## 4. VolReg loss (numerically sound)

Proper covariance over the $N$ spatial samples, symmetric eigensolver `eigvalsh`
(no SVD back-prop instability), trace-relative jitter, single clamp. The objective is
still the *variance of* $\log\det\Sigma$ across the mini-batch — i.e. a target-free
volume-homogeneity prior, kept intentionally so its (now mechanistic) negative result
can be reported cleanly.

In [ ]:
class VolumeRegularizationLoss(nn.Module):
    def __init__(self, eps=1e-4):
        super().__init__(); self.eps = eps
    def forward(self, b):
        B, C, D, H, W = b.shape; N = D*H*W
        X = b.reshape(B, C, N); Xc = X - X.mean(-1, keepdim=True)
        cov = torch.bmm(Xc, Xc.transpose(1, 2)) / (N - 1)
        tau = torch.diagonal(cov, dim1=-2, dim2=-1).mean(-1).view(B, 1, 1)
        I = torch.eye(C, device=b.device).unsqueeze(0)
        ev = torch.linalg.eigvalsh(cov + self.eps*tau*I).clamp_min(1e-8)
        ld = torch.log(ev).sum(1)
        return ((ld - ld.mean())**2).mean()

## 5. Architectures (backbone-constant ablation) — both return `(logits, bottleneck)`

In [ ]:
def cbr(i, o): return nn.Sequential(nn.Conv3d(i, o, 3, padding=1), nn.BatchNorm3d(o), nn.ReLU())

class UNet3D(nn.Module):
    def __init__(self, out_ch=3, use_lop=False, gamma=1e-2, mode='whiten', width=(16,32,64)):
        super().__init__(); self.use_lop = use_lop; c1,c2,c3 = width
        self.e1 = cbr(1, c1); self.e2 = nn.Sequential(nn.MaxPool3d(2), cbr(c1, c2))
        self.bott = nn.Sequential(nn.MaxPool3d(2), cbr(c2, c3))
        if use_lop: self.lop = LatentDecorrelationBlock(c3, gamma, mode)
        self.u2 = nn.ConvTranspose3d(c3, c2, 2, stride=2); self.d2 = cbr(c2*2, c2)
        self.u1 = nn.ConvTranspose3d(c2, c1, 2, stride=2); self.d1 = cbr(c1*2, c1)
        self.final = nn.Conv3d(c1, out_ch, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(e1); b = self.bott(e2)
        if self.use_lop: b = self.lop(b)
        d2 = self.d2(torch.cat([self.u2(b), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.final(d1), b

class ResUNet(nn.Module):
    def __init__(self, out_ch=3, use_lop=False, gamma=1e-2, mode='whiten', width=(16,32,64)):
        super().__init__(); self.use_lop = use_lop; c1,c2,c3 = width
        self.e1 = cbr(1, c1); self.r1 = cbr(c1, c1)
        self.e2 = nn.Sequential(nn.MaxPool3d(2), cbr(c1, c2)); self.r2 = cbr(c2, c2)
        self.bott = nn.Sequential(nn.MaxPool3d(2), cbr(c2, c3))
        if use_lop: self.lop = LatentDecorrelationBlock(c3, gamma, mode)
        self.u2 = nn.ConvTranspose3d(c3, c2, 2, stride=2); self.d2 = cbr(c2*2, c2)
        self.u1 = nn.ConvTranspose3d(c2, c1, 2, stride=2); self.d1 = cbr(c1*2, c1)
        self.final = nn.Conv3d(c1, out_ch, 1)
    def forward(self, x):
        h = self.e1(x); e1 = F.relu(h + self.r1(h))
        e2 = self.e2(e1); e2 = F.relu(e2 + self.r2(e2))
        b = self.bott(e2)
        if self.use_lop: b = self.lop(b)
        d2 = self.d2(torch.cat([self.u2(b), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.final(d1), b

## 6. Segmentation loss, metrics (Dice / ASSD / ECE)

In [ ]:
class DiceCELoss(nn.Module):
    def __init__(self): super().__init__(); self.ce = nn.CrossEntropyLoss()
    def forward(self, logits, target):
        ce = self.ce(logits, target); p = F.softmax(logits, 1)
        oh = F.one_hot(target, p.shape[1]).permute(0, 4, 1, 2, 3).float()
        dice = 1 - ((2*(p*oh).sum((2,3,4)) + 1)/(p.sum((2,3,4)) + oh.sum((2,3,4)) + 1)).mean()
        return ce + dice

def dice_score(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool); s = pred.sum() + gt.sum()
    return 1.0 if s == 0 else 2.0*(pred & gt).sum()/s

def assd(pred, gt, spacing=(1,1,1)):
    pred, gt = pred.astype(bool), gt.astype(bool)
    if pred.sum() == 0 or gt.sum() == 0:
        return 16.0  # capped sentinel; reported separately from finite cases
    if HAVE_MEDPY:
        return float(_medpy_assd(pred, gt, voxelspacing=spacing))
    def sd(a, b):
        edge = a ^ ndimage.binary_erosion(a)
        return ndimage.distance_transform_edt(~b, sampling=spacing)[edge]
    return float(np.mean(np.concatenate([sd(pred, gt), sd(gt, pred)])))

def compute_ece(probs, labels, n_bins=15):
    conf, pred = probs.max(1); acc = (pred == labels).float()
    bins = torch.linspace(0, 1, n_bins + 1); ece = 0.0
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i+1])
        if m.any(): ece += m.float().mean()*(acc[m].mean() - conf[m].mean()).abs()
    return float(ece)

## 7. Training

In [ ]:
def train_model(model, data, use_volreg=False):
    model.to(cfg.device).train()
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    seg, vr = DiceCELoss(), VolumeRegularizationLoss()
    for ep in range(cfg.epochs):
        idx = np.random.default_rng(ep).permutation(len(data))
        for s in range(0, len(data), cfg.batch_size):
            bi = idx[s:s+cfg.batch_size]
            xb = torch.stack([torch.from_numpy(data[i][0]) for i in bi]).unsqueeze(1).float().to(cfg.device)
            yb = torch.stack([torch.from_numpy(data[i][1]) for i in bi]).long().to(cfg.device)
            opt.zero_grad(); logits, b = model(xb); loss = seg(logits, yb)
            if use_volreg: loss = loss + cfg.alpha_volreg*vr(b)
            loss.backward(); opt.step()
    return model.eval()

## 8. Run experiments

Backbone-constant ablation on each backbone:
`Baseline` · `LOP(whiten)` · `LOP(standardize)` (component control) · `VolReg` (ResUNet).
For the real run, add the `gamma_list` sweep for the whitening models.

In [ ]:
NETS = {}
for arch, Net in [('ResUNet', ResUNet), ('UNet3D', UNet3D)]:
    NETS[f'Baseline_{arch}']  = train_model(Net(), train_set)
    NETS[f'LOP_{arch}']       = train_model(Net(use_lop=True, mode='whiten',     gamma=1e-3), train_set)
    NETS[f'STD_{arch}']       = train_model(Net(use_lop=True, mode='standardize', gamma=1e-3), train_set)
NETS['VolReg_ResUNet'] = train_model(ResUNet(), train_set, use_volreg=True)

# gamma sweep (whitening only) — drives the conditioning diagnostic
for arch, Net in [('ResUNet', ResUNet), ('UNet3D', UNet3D)]:
    for g in cfg.gamma_list:
            NETS[f'LOPg_{arch}_{g:.0e}'] = train_model(Net(use_lop=True, mode='whiten', gamma=g), train_set)
print('trained:', list(NETS))

## 9. Inference + degradation sweep + **true** condition-number logging

In [ ]:
scenarios = [('noise', n, f'Noise_{n:.2f}'.replace('.', '_')) for n in cfg.noise_levels] + \
            [('blur',  b, f'Blur_{b:.2f}'.replace('.', '_'))  for b in cfg.blur_sigmas]
rng = np.random.default_rng(cfg.seed)
metric_rows, cond_rows, ece_rows = [], [], []

for name, model in NETS.items():
    arch = 'ResUNet' if 'ResUNet' in name else 'UNet3D'
    for kind, sev, label in scenarios:
        if sev == 0.0: label = 'Ideal'
        all_prob, all_lab = [], []
        for vi, (img, lab) in enumerate(test_set):
            ic = corrupt_volume(img, kind, sev, rng) if sev > 0 else img
            xt = torch.from_numpy(ic).float().unsqueeze(0).unsqueeze(0).to(cfg.device)
            with torch.no_grad():
                logits, b = model(xt)
                kappa = None
                if getattr(model, 'use_lop', False) and model.lop.mode == 'whiten':
                    _, kappa = model.lop(b, return_cond=True)
                prob = F.softmax(logits, 1)[0].cpu()          # (Cls,D,H,W)
                pred = prob.argmax(0).numpy()
            for cls, struct in [(1, 'Anterior'), (2, 'Posterior')]:
                metric_rows.append(dict(Model=name, Arch=arch, Scenario=label, Structure=struct,
                                        Volume=vi, Dice=dice_score(pred == cls, lab == cls),
                                        ASSD=assd(pred == cls, lab == cls)))
            if kappa is not None:
                cond_rows.append(dict(Model=name, Scenario=label, Volume=vi, Condition_Number=kappa,
                                      ASSD_avg=np.mean([assd(pred == c, lab == c) for c in (1, 2)])))
            all_prob.append(prob.permute(1, 2, 3, 0).reshape(-1, prob.shape[0]))
            all_lab.append(torch.from_numpy(lab).reshape(-1))
        ece_rows.append(dict(Model=name, Scenario=label,
                             ECE=compute_ece(torch.cat(all_prob), torch.cat(all_lab))))
metrics_df = pd.DataFrame(metric_rows)
ece_df = pd.DataFrame(ece_rows)
cond_df = pd.DataFrame(cond_rows)
print('metric rows:', len(metrics_df), '| cond rows:', len(cond_df), '| ece rows:', len(ece_df))
metrics_df.head()

## 10. Statistics — paired tests, effect sizes, **multiple-comparison correction**

For every (scenario × structure × backbone) we pair `LOP(whiten)` against `Baseline` on the
same volumes: Wilcoxon signed-rank $p$, paired Cohen $d$, and bootstrap median CIs. Holm and
Benjamini–Hochberg are applied **across the whole family** so borderline single-test $p$-values
are judged honestly.

In [ ]:
# ---- helpers ----
def cohen_d_paired(a, b):
    d = np.asarray(a) - np.asarray(b); sd = d.std(ddof=1)
    return 0.0 if sd == 0 else d.mean()/sd
def holm(p):
    p = np.asarray(p); m = len(p); o = np.argsort(p); adj = np.empty(m); run = 0.0
    for r, idx in enumerate(o): run = max(run, (m - r)*p[idx]); adj[idx] = min(run, 1.0)
    return adj
def benjamini_hochberg(p):
    p = np.asarray(p); m = len(p); o = np.argsort(p); adj = np.empty(m); prev = 1.0
    for r in range(m - 1, -1, -1):
        idx = o[r]; prev = min(prev, p[idx]*m/(r + 1)); adj[idx] = min(prev, 1.0)
    return adj
def paired_p(a, b):
    a, b = np.asarray(a), np.asarray(b)
    if np.allclose(a, b): return 1.0
    try: return float(wilcoxon(a, b)[1])
    except Exception: return 1.0
def p5(x): return float(np.percentile(np.asarray(x), 5))
def split_assd(vals, cap=16.0):
    """Return (median over FINITE ASSD, failure_rate) — capped values are failures."""
    v = np.asarray(vals, float); fail = v >= cap; fin = v[~fail]
    med = float(np.median(fin)) if fin.size else float('nan')
    return med, float(fail.mean())

# ---- unified paired comparison A vs B (A is the 'treatment') ----
def compare(df, a_model, b_model, arch, tag):
    rows = []
    sub = df[df.Arch == arch]
    for (sc, st), g in sub.groupby(['Scenario', 'Structure']):
        A = g[g.Model == a_model].sort_values('Volume')
        B = g[g.Model == b_model].sort_values('Volume')
        if len(A) == 0 or len(B) == 0: continue
        mA, fA = split_assd(A.ASSD.values); mB, fB = split_assd(B.ASSD.values)
        rows.append(dict(Comparison=tag, Scenario=sc, Structure=st, Arch=arch,
            Dice_A=A.Dice.median(), Dice_B=B.Dice.median(),
            P5_A=p5(A.Dice.values), P5_B=p5(B.Dice.values),
            d_Dice=cohen_d_paired(A.Dice.values, B.Dice.values),
            p_Dice=paired_p(A.Dice.values, B.Dice.values),
            ASSDfin_A=mA, ASSDfin_B=mB, FailRate_A=fA, FailRate_B=fB,
            p_ASSD=paired_p(A.ASSD.values, B.ASSD.values)))
    r = pd.DataFrame(rows)
    if len(r):
        r['p_Dice_holm'] = holm(r.p_Dice.values)
        r['p_Dice_BH']   = benjamini_hochberg(r.p_Dice.values)
    return r

comparisons = []
for arch in ['ResUNet', 'UNet3D']:
    comparisons.append(compare(metrics_df, f'LOP_{arch}', f'Baseline_{arch}', arch, 'whiten_vs_baseline'))
    comparisons.append(compare(metrics_df, f'LOP_{arch}', f'STD_{arch}',      arch, 'whiten_vs_std'))
    comparisons.append(compare(metrics_df, f'STD_{arch}', f'Baseline_{arch}', arch, 'std_vs_baseline'))
report = pd.concat([c for c in comparisons if len(c)], ignore_index=True)
report.to_csv('report_pairwise_v2.csv', index=False)

def survivors(tag):
    r = report[report.Comparison == tag]
    sig = r[r.p_Dice_holm < 0.05]
    return len(sig), len(r)
for tag in ['whiten_vs_baseline', 'whiten_vs_std', 'std_vs_baseline']:
    s, n = survivors(tag)
    print(f'{tag:22s}: Dice significant after Holm = {s}/{n}')
print()
print('Key question -> if whiten_vs_std has FEW survivors, the orthogonalization '
      'does not beat plain standardization (rename the contribution).')
report[report.Comparison == 'whiten_vs_std'].round(4)

## 11. Diagnostics & plots

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
# (a) Conditioning vs boundary error — Spearman on FINITE ASSD only (capped points removed),
#     because empty-mask failures (=16) would otherwise inflate the correlation.
if len(cond_df):
    fin = cond_df[cond_df.ASSD_avg < 16.0]
    if len(fin) > 10 and fin.Condition_Number.nunique() > 3:
        rho, pr = spearmanr(np.log10(fin.Condition_Number.values), fin.ASSD_avg.values)
        plt.figure(figsize=(6,4))
        plt.scatter(np.log10(fin.Condition_Number.values), fin.ASSD_avg.values, alpha=.35, s=16)
        plt.xlabel(r'$\log_{10}\ \kappa(\Sigma_s)$  (TRUE condition number)')
        plt.ylabel('Average ASSD (mm), finite only')
        plt.title(f'Conditioning vs boundary error (finite cases)\nSpearman rho={rho:.3f} (p={pr:.2g})')
        plt.tight_layout(); plt.savefig('cond_vs_assd_v2.png', dpi=130); plt.show()
        print('CONFOUND: kappa largely indexes the gamma level (kappa ~ 1/gamma across the sweep),')
        print('so this mostly says "less shrinkage -> worse boundaries". See failure-vs-gamma below.')
else:
    print('No condition-number records (whitening models needed).')

In [ ]:
# (b) The cleaner conditioning diagnostic: ASSD failure rate vs shrinkage gamma.
sweep = metrics_df[metrics_df.Model.str.startswith('LOPg_')].copy()
if len(sweep):
    import re
    sweep['gamma'] = sweep.Model.str.extract(r'_(1e-0\d)$')[0].astype(float)
    sweep['arch']  = np.where(sweep.Model.str.contains('ResUNet'), 'ResUNet', 'UNet3D')
    sweep['fail']  = (sweep.ASSD >= 16.0).astype(float)
    tab = sweep.groupby(['arch', 'gamma']).fail.mean().mul(100).round(2).reset_index()
    print('ASSD failure rate (%) by shrinkage gamma (smaller gamma = more aggressive whitening):')
    print(tab.pivot(index='gamma', columns='arch', values='fail').to_string())
    fig, ax = plt.subplots(figsize=(6,4))
    for arch, g in tab.groupby('arch'):
        ax.plot(-np.log10(g.gamma), g.fail, marker='o', label=arch)
    ax.set_xlabel(r'$-\log_{10}\gamma$  (more whitening $\rightarrow$)'); ax.set_ylabel('ASSD failure rate (%)')
    ax.set_title('More aggressive whitening $\\Rightarrow$ more boundary failures'); ax.legend()
    plt.tight_layout(); plt.savefig('failrate_vs_gamma_v2.png', dpi=130); plt.show()
else:
    print('Run the gamma sweep (default on) to populate failure-vs-gamma.')

In [ ]:
# VolReg dissociation: Anterior vs Posterior (ResUNet), VolReg vs Baseline.
vr = compare(metrics_df, 'VolReg_ResUNet', 'Baseline_ResUNet', 'ResUNet', 'volreg_vs_baseline')
if len(vr):
    show = vr[['Scenario','Structure','Dice_B','Dice_A','d_Dice','p_Dice_holm',
               'ASSDfin_B','ASSDfin_A','FailRate_A','p_ASSD']].rename(
               columns={'Dice_B':'Dice_base','Dice_A':'Dice_volreg',
                        'ASSDfin_B':'ASSD_base','ASSDfin_A':'ASSD_volreg'})
    show.to_csv('volreg_dissociation_v2.csv', index=False)
    print('VolReg vs Baseline — expect Anterior DOWN (d<0) but Posterior often UP (esp. ASSD):')
    display(show.round(4))
else:
    print('VolReg model not found.')

In [ ]:
# (b) Robustness curve: median Dice (LOP vs Baseline) vs noise level
noise_df = metrics_df[metrics_df.Scenario.str.startswith(('Ideal','Noise'))].copy()
def sev(s): return 0.0 if s == 'Ideal' else float(s.replace('Noise_','').replace('_','.'))
noise_df['noise'] = noise_df.Scenario.map(sev)
fig, ax = plt.subplots(figsize=(6,4))
for arch in ['ResUNet','UNet3D']:
    for model, ls in [(f'Baseline_{arch}','--'), (f'LOP_{arch}','-')]:
        sub = noise_df[noise_df.Model == model].groupby('noise').Dice.median()
        if len(sub): ax.plot(sub.index, sub.values, ls, marker='o', label=model)
ax.set_xlabel('Noise level (std)'); ax.set_ylabel('Median Dice'); ax.legend(fontsize=7)
ax.set_title('Robustness: LOP(whiten) vs Baseline'); plt.tight_layout()
plt.savefig('robustness_clean.png', dpi=130); plt.show()

## 12. How to read these results (established vs hypothesis)

- **VolReg** is reported as a *mechanistic negative result*: the loss penalises the variance of
  $\log\det\Sigma$, a target-free volume-homogeneity prior dominated by the smallest eigenvalue,
  so it sacrifices rare/heterogeneous structures. The numerically clean implementation removes
  the "it's just a numerical bug" objection.

- **The decorrelation block** is only a contribution if `LOP(whiten)` beats both `Baseline`
  **and** `STD` (standardize-only) after Holm/BH correction. If `whiten ≈ STD`, the effect is
  normalization + channel attention, not orthogonalization — and the method must be renamed
  accordingly. The `mode` ablation answers this directly.

- **The conditioning hypothesis** is now testable: with the $\gamma$ sweep, a positive Spearman
  $\rho$ between the *true* $\kappa(\Sigma_s)$ and ASSD would support "ill-conditioned whitening
  degrades boundaries"; a flat/zero $\rho$ would refute it. Either way the figure plots a real
  matrix property, not $1/\gamma$ in disguise.

- Report **effect sizes and corrected $p$-values**, capped-ASSD (=16) cases separately from
  finite ones, and the worst-case (P5) Dice rather than only the median.

**v2 reading rule.** The contribution is the orthogonalization *only if* `whiten_vs_std` shows survivors after Holm across many cells. If it does not (few/Anterior-only, with failures under stress), report: (1) VolReg as an Anterior/Posterior **dissociation**; (2) **standardization + gating** as the real, modest benefit; (3) whitening as a **secondary negative result** explained by the failure-vs-$\gamma$ trend. Always show failure rate beside finite-ASSD medians.

In [ ]:
# Export elsarticle-friendly LaTeX tables (booktabs) from the v2 reports.
def df_to_latex(df, caption, label, float_cols=None, fmt='%.3f'):
    d = df.copy()
    if float_cols:
        for c in float_cols:
            if c in d: d[c] = d[c].map(lambda x: (fmt % x) if pd.notnull(x) else '--')
    body = d.to_latex(index=False, escape=True, column_format='l'*d.shape[1])
    return (f"\\begin{{table}}[t]\n\\centering\n\\caption{{{caption}}}\n"
            f"\\label{{{label}}}\n{body}\\end{{table}}\n")

ws = report[report.Comparison == 'whiten_vs_std'][
        ['Scenario','Structure','Arch','d_Dice','p_Dice_holm','FailRate_A','FailRate_B']]
tex = df_to_latex(ws, 'Ablation: latent whitening vs standardization (paired, Holm-corrected). '
                      'FailRate\\_A = whitening, FailRate\\_B = standardization.',
                  'tab:whiten_vs_std', float_cols=['d_Dice','p_Dice_holm','FailRate_A','FailRate_B'])
open('table_whiten_vs_std.tex','w').write(tex)
print(tex[:700], '\n...\nWrote table_whiten_vs_std.tex')

## 13. Multi-seed stability harness (the reproducibility discipline)

Single-run effect sizes at $N\approx26$ are dominated by a handful of empty-mask
failures, so the same cell can swing from $d=-4.5$ to $d=+3.1$ between trainings.
**Rule:** freeze a fixed seed-set, train every configuration on all seeds, and only
report a directional effect if its **sign is stable across seeds**. Failures are a
separate binomial outcome (rate + Wilson CI), never mixed into the Dice effect size.

Workflow: `build_suite(seed)` trains the full model set for one seed,
`eval_suite` scores it under every corruption, `compare_seeded` aggregates the paired
Cohen $d$ across seeds (bootstrap CI of the mean + a `sign_stable` flag) and pools the
failure rate with a Wilson interval. Write the manuscript's numbers from this table —
nothing else.

In [ ]:
import math
N_SEEDS = 3 if SMOKE else 5          # use >=5 for the manuscript
FROZEN_SEEDS = list(range(N_SEEDS))  # the fixed seed-set; never cherry-pick

def wilson_ci(k, n, z=1.96):
    if n == 0: return (float('nan'), float('nan'))
    p = k/n; den = 1 + z*z/n
    c = (p + z*z/(2*n))/den
    h = (z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)))/den
    return max(0.0, c-h), min(1.0, c+h)

def build_suite(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    nets = {}
    for arch, Net in [('ResUNet', ResUNet), ('UNet3D', UNet3D)]:
        nets[f'Baseline_{arch}'] = train_model(Net(), train_set)
        nets[f'LOP_{arch}']      = train_model(Net(use_lop=True, mode='whiten',     gamma=1e-3), train_set)
        nets[f'STD_{arch}']      = train_model(Net(use_lop=True, mode='standardize', gamma=1e-3), train_set)
    nets['VolReg_ResUNet'] = train_model(ResUNet(), train_set, use_volreg=True)
    return nets

def eval_suite(nets, seed):
    rng = np.random.default_rng(1000 + seed); rows = []
    for name, model in nets.items():
        arch = 'ResUNet' if 'ResUNet' in name else 'UNet3D'
        for kind, sev, label in scenarios:
            lab = 'Ideal' if sev == 0.0 else label
            for vi, (img, lb) in enumerate(test_set):
                ic = corrupt_volume(img, kind, sev, rng) if sev > 0 else img
                xt = torch.from_numpy(ic).float().unsqueeze(0).unsqueeze(0).to(cfg.device)
                with torch.no_grad():
                    logits, _ = model(xt); pred = logits.argmax(1)[0].cpu().numpy()
                for cls, struct in [(1, 'Anterior'), (2, 'Posterior')]:
                    rows.append(dict(Seed=seed, Model=name, Arch=arch, Scenario=lab,
                                     Structure=struct, Volume=vi,
                                     Dice=dice_score(pred == cls, lb == cls),
                                     ASSD=assd(pred == cls, lb == cls)))
    return rows

all_rows = []
for s in FROZEN_SEEDS:
    print(f'  seed {s} ...', end='', flush=True)
    all_rows += eval_suite(build_suite(s), s); print(' done')
metrics_seeds = pd.DataFrame(all_rows)
metrics_seeds.to_csv('metrics_seeds.csv', index=False)
print('seeded metric rows:', len(metrics_seeds), '| seeds:', sorted(metrics_seeds.Seed.unique()))

In [ ]:
def compare_seeded(df, a_model, b_model, arch, tag, min_abs_d=0.2):
    """Aggregate paired Cohen d across seeds; flag sign-stable effects; pool failures (Wilson)."""
    rows = []; sub = df[df.Arch == arch]
    for (sc, st), g in sub.groupby(['Scenario', 'Structure']):
        ds = []; fA = fB = nA = nB = 0
        for seed, gs in g.groupby('Seed'):
            A = gs[gs.Model == a_model].sort_values('Volume')
            B = gs[gs.Model == b_model].sort_values('Volume')
            if len(A) == 0 or len(A) != len(B): continue
            ds.append(cohen_d_paired(A.Dice.values, B.Dice.values))
            fA += int((A.ASSD.values >= 16).sum()); nA += len(A)
            fB += int((B.ASSD.values >= 16).sum()); nB += len(B)
        if len(ds) < 2: continue
        ds = np.array(ds); dm = float(ds.mean())
        rng = np.random.default_rng(0)
        bs = [rng.choice(ds, len(ds), replace=True).mean() for _ in range(2000)]
        lo, hi = float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))
        frac_same = float(max((ds > 0).mean(), (ds < 0).mean()))
        # CORRECTED criterion: across-seed CI of mean d excludes zero (unanimity of sign
        # is too strict at higher n and hides real effects). Require CI-excludes-0,
        # a >=70% sign-agreement sanity check, and a minimum magnitude.
        sign_stable = bool((lo > 0 or hi < 0) and frac_same >= 0.7 and abs(dm) >= min_abs_d)
        strong = bool(sign_stable and abs(dm) >= 0.4)
        flA, fhA = wilson_ci(fA, nA); flB, fhB = wilson_ci(fB, nB)
        rows.append(dict(Comparison=tag, Scenario=sc, Structure=st, Arch=arch, n_seeds=len(ds),
            d_mean=round(dm, 3), d_lo=round(lo, 3), d_hi=round(hi, 3),
            frac_same_sign=round(frac_same, 2), SIGN_STABLE=sign_stable, STRONG=strong,
            failA=round(fA/nA, 3), failA_lo=round(flA, 3), failA_hi=round(fhA, 3),
            failB=round(fB/nB, 3), failB_lo=round(flB, 3), failB_hi=round(fhB, 3)))
    return pd.DataFrame(rows)

seeded = []
for arch in ['ResUNet', 'UNet3D']:
    seeded.append(compare_seeded(metrics_seeds, f'LOP_{arch}', f'Baseline_{arch}', arch, 'whiten_vs_baseline'))
    seeded.append(compare_seeded(metrics_seeds, f'LOP_{arch}', f'STD_{arch}',      arch, 'whiten_vs_std'))
    seeded.append(compare_seeded(metrics_seeds, f'STD_{arch}', f'Baseline_{arch}', arch, 'std_vs_baseline'))
seeded.append(compare_seeded(metrics_seeds, 'VolReg_ResUNet', 'Baseline_ResUNet', 'ResUNet', 'volreg_vs_baseline'))
seeded_report = pd.concat([s for s in seeded if len(s)], ignore_index=True)
seeded_report.to_csv('report_seeded.csv', index=False)

print('Sign-STABLE directional effects (the only ones safe to claim):')
for tag in seeded_report.Comparison.unique():
    r = seeded_report[seeded_report.Comparison == tag]
    st = r[r.SIGN_STABLE]
    print(f'  {tag:20s}: {len(st)}/{len(r)} cells sign-stable '
          f'(+{sum(st.d_mean>0)} / -{sum(st.d_mean<0)})')
print('\\nEvery manuscript claim must cite a SIGN_STABLE=True row from report_seeded.csv.')
seeded_report[seeded_report.SIGN_STABLE].round(3)

### Reading `report_seeded.csv`

- **`SIGN_STABLE=True`** = the across-seed bootstrap CI of the mean $d$ excludes zero
  (with $\ge70\%$ sign agreement and $|d|\ge0.2$). **`STRONG=True`** adds $|d|\ge0.4$ —
  these are the headline-grade effects. Requiring unanimous sign on every seed is *too strict*
  at $n\ge10$ and hides real effects, so it is not used.
- Cells that are significant in one run but flip or straddle zero across seeds are **reported
  as inconclusive**, not as findings.
- Quote failure as `failA [failA_lo, failA_hi]` (Wilson), separately from $d$.
- Decide the thesis from this table: if `whiten_vs_std` has no stable positive cells, the
  contribution is standardization; if the architectural split is stable, that becomes the claim.
  Either way, the abstract is written *after* this cell, from this cell.

# ***Figure***

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load dataframe from metrics_seeds.csv
metrics_df = pd.read_csv('/content/drive/MyDrive/LOP/metrics_seeds.csv')

# Filter for Ideal and Noise scenarios
noise_df = metrics_df[metrics_df.Scenario.str.startswith(('Ideal', 'Noise'))].copy()

# Map scenario string to numeric noise level values
def sev(s):
    return 0.0 if s == 'Ideal' else float(s.replace('Noise_', '').replace('_', '.'))

noise_df['noise'] = noise_df.Scenario.map(sev)

# Plot setup
fig, ax = plt.subplots(figsize=(7, 4.5))

# Define architectures, models, and line styles
architectures = ['ResUNet', 'UNet3D']
model_types = [
    ('Baseline', ':'),  # Dotted line for Baseline
    ('STD', '--'),       # Dashed line for Standard
    ('LOP', '-')         # Solid line for LOP
]

# Loop through architectures and models to plot curves
for arch in architectures:
    for prefix, ls in model_types:
        model_name = f'{prefix}_{arch}'

        # Group by noise level and compute the median Dice score
        sub = noise_df[noise_df.Model == model_name].groupby('noise').Dice.median()

        # Plot if data exists for the given combination
        if len(sub):
            ax.plot(sub.index, sub.values, ls, marker='o', label=model_name)

# Formatting graph aesthetics
ax.set_xlabel('Noise level (std)')
ax.set_ylabel('Median Dice')
ax.legend(fontsize=8, loc='lower left')
ax.set_title('Robustness Curve: Baseline vs STD vs LOP')
plt.tight_layout()

# Save and show the plot
plt.savefig('robustness_comparison_all_models.png', dpi=600)
plt.show()